In [6]:
import pandas as pd
import numpy as np
import os
os.getcwd()

'C:\\Users\\Matt\\Documents\\USD_JPY_Capstone\\Data_Wrangling\\notebooks'

In [7]:
old = pd.read_csv("../data_raw/jp_cpi_index.csv")
new = pd.read_csv("../data_raw/jp_cpi_estat_2020base.csv", encoding='shift_jis', header=None, skiprows=6, names=['yyyymm','value'])

In [8]:
new.head()

,yyyymm,value
0,194601,NaN
1,194602,NaN
2,194603,NaN
3,194604,NaN
4,194605,NaN


In [9]:
old.head()

,observation_date,JPNCPIALLMINMEI
0,1955-01-01,17.09859
1,1955-02-01,17.09859
2,1955-03-01,16.98852
3,1955-04-01,17.13528
4,1955-05-01,16.97017


In [12]:
old.columns

Index(['observation_date', 'JPNCPIALLMINMEI'], dtype='object')

In [21]:
old.columns = ['DATE', 'jp_cpi_old']

In [22]:
old.columns

Index(['DATE', 'jp_cpi_old'], dtype='object')

In [23]:
old['DATE'] = pd.to_datetime(old['DATE'])

In [24]:
old.head()

,DATE,jp_cpi_old
0,1955-01-01,17.09859
1,1955-02-01,17.09859
2,1955-03-01,16.98852
3,1955-04-01,17.13528
4,1955-05-01,16.97017


In [25]:
new['DATE'] = pd.to_datetime(new['yyyymm'].astype(str), format='%Y%m')
new = new[['DATE', 'value']].rename(columns={'value': 'jp_cpi_new'})

In [28]:
new.head(), new.tail()

(        DATE  jp_cpi_new
 0 1946-01-01         NaN
 1 1946-02-01         NaN
 2 1946-03-01         NaN
 3 1946-04-01         NaN
 4 1946-05-01         NaN,
           DATE  jp_cpi_new
 960 2026-01-01       115.1
 961 2026-02-01       114.3
 962 2026-03-01       114.9
 963 2026-04-01       115.3
 964 2026-05-01       115.9)

In [27]:
merged_check = old.merge(new, on='DATE', how='inner')
merged_check.tail(10)

,DATE,jp_cpi_old,jp_cpi_new
788,2020-09-01,101.7053,99.9
789,2020-10-01,101.6035,99.8
790,2020-11-01,101.2980,99.4
791,2020-12-01,101.0944,99.2
792,2021-01-01,101.6035,99.8
793,2021-02-01,101.6035,99.7
794,2021-03-01,101.7053,99.9
795,2021-04-01,100.8908,98.9
796,2021-05-01,101.1962,99.2
797,2021-06-01,101.2980,99.4


In [29]:
splice_date = pd.Timestamp('2021-06-01')

old_val = merged_check.loc[merged_check['DATE'] == splice_date, 'jp_cpi_old'].values[0]
new_val = merged_check.loc[merged_check['DATE'] == splice_date, 'jp_cpi_new'].values[0]

scale_factor = old_val / new_val
scale_factor

np.float64(1.0190945674044265)

In [30]:
new['jp_cpi_rescaled'] = new['jp_cpi_new'] * scale_factor

jp_cpi_final = pd.concat([
    old.loc[old['DATE'] <= splice_date, ['DATE', 'jp_cpi_old']].rename(columns={'jp_cpi_old': 'jp_cpi_index'}),
    new.loc[new['DATE'] > splice_date, ['DATE', 'jp_cpi_rescaled']].rename(columns={'jp_cpi_rescaled': 'jp_cpi_index'})
]).reset_index(drop=True)

jp_cpi_final.tail(10)

,DATE,jp_cpi_index
847,2025-08-01,116.380600
848,2025-09-01,116.278690
849,2025-10-01,117.195875
850,2025-11-01,117.705423
851,2025-12-01,117.501604
852,2026-01-01,117.297785
853,2026-02-01,116.482509
854,2026-03-01,117.093966
855,2026-04-01,117.501604
856,2026-05-01,118.113060


In [32]:
os.makedirs('data_processed', exist_ok=True)
jp_cpi_final.to_csv('../data_processed/jp_cpi_extended.csv', index=False)